# 05 — Train ConvNeXt-Tiny stretch model

Prereqs: run `01_colab_setup.ipynb` first so deps are installed and `data/fer2013/` is populated.

**Modern CNN stretch baseline** for the comparison study. Hooks to EmoNeXt (`report.md:488`) which uses ConvNeXt as its backbone. Unlike DAN, this needs **no third-party repo** — torchvision provides the architecture and ImageNet weights.

Target: WAR ~72-74% on FER-2013 PrivateTest (above DAN's expected ~70-72%; human upper bound ~65-68% per `report.md:1928`).

This notebook:
1. Runs `python -m src.train --config configs/convnext_fer2013.yaml`.
2. Evaluates `runs/convnext_fer2013/best.pth` on the FER-2013 PrivateTest split.
3. Displays the confusion matrix.

In [ ]:
# === Bootstrap: mount Drive, clone repo, hydrate data, link runs/ to Drive ===
# Requires Colab "Secrets" entry GH_TOKEN with a fine-grained PAT for radudeaconu/fer.
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
if not Path('/content/fer').exists():
    os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
    !git clone https://$GH_TOKEN@github.com/radudeaconu/fer.git /content/fer
%cd /content/fer
!pip install -q -r requirements.txt
%run scripts/colab_bootstrap.py

import torch
print('CUDA:', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


## Train (≈65 min on T4 for 30 epochs)

In [ ]:
!python -m src.train --config configs/convnext_fer2013.yaml

## Evaluate

In [ ]:
!python -m src.eval --config configs/convnext_fer2013.yaml --ckpt runs/convnext_fer2013/best.pth

In [ ]:
from IPython.display import Image
Image('runs/convnext_fer2013/eval/confusion_matrix.png')

## Compare to DAN at a glance
Run this cell after both `02_train_dan.ipynb` and this notebook have completed.

In [ ]:
import json
from pathlib import Path
rows = []
for name, p in [('DAN', 'runs/dan_fer2013/eval/metrics.json'),
                ('ConvNeXt-Tiny', 'runs/convnext_fer2013/eval/metrics.json')]:
    p = Path(p)
    if not p.exists():
        print(f'{name}: {p} not found yet'); continue
    m = json.loads(p.read_text())
    rows.append({'model': name, 'WAR': m['war'], 'UAR': m['uar']})
import pandas as pd
pd.DataFrame(rows)